# 01 — Crash EDA: Chicago High-Crash Corridor Panel

**Project:** Chicago Road Safety Investment Prioritizer  
**Aligned with:** City of Chicago Vision Zero goals  
**Type:** Read-only exploratory analysis — no source files are modified.  
**Grain:** Corridor-month (43 corridors × 96 months = 4,128 rows, 2018-01 to 2025-12)  

---

Every printed number in this notebook is **computed from the data in the cell
immediately above**. No figures are hardcoded.


In [ ]:
from __future__ import annotations
import json
from pathlib import Path
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import matplotlib.patches as mpatches
import numpy as np
import pandas as pd

%matplotlib inline
plt.rcParams.update({
    'figure.dpi': 110,
    'axes.spines.top': False,
    'axes.spines.right': False,
    'axes.titlesize': 12,
    'axes.labelsize': 10,
})

ROOT = Path('.').resolve()
if ROOT.name == 'notebooks':
    ROOT = ROOT.parent
print('Project root:', ROOT)


---
## Section 1 — Purpose & Data Summary


In [ ]:
PANEL_PATH    = ROOT / 'data' / 'processed' / 'corridor_month_panel.parquet'
REGISTER_PATH = ROOT / 'data' / 'interim'   / 'high_crash_corridor_register.csv'
GEO_PATH      = ROOT / 'docs' / 'data_quality' / 'corridor_geometry_validation.json'

df       = pd.read_parquet(PANEL_PATH)
register = pd.read_csv(REGISTER_PATH)
with open(GEO_PATH) as fh:
    geo = json.load(fh)

n_rows      = len(df)
n_corridors = df['corridor_id'].nunique()
month_min   = df['crash_month_start'].min().strftime('%Y-%m')
month_max   = df['crash_month_start'].max().strftime('%Y-%m')
geo_status  = geo['status']
geo_crit    = geo['summary']['critical_failures']

SEV_COLS = [
    'fatal_crashes', 'serious_injury_crashes',
    'moderate_injury_crashes', 'minor_injury_crashes',
    'property_damage_only_crashes', 'unknown_severity_crashes',
]

print('Panel Summary')
print('  Source     :', PANEL_PATH.relative_to(ROOT))
print('  Rows       :', f'{n_rows:,}')
print('  Corridors  :', n_corridors)
print('  Month range:', month_min, 'to', month_max)
print('  Columns    :', len(df.columns))
print('  Sev cols   :', SEV_COLS)
print()
print('Geometry validation status :', geo_status)
print('Critical failures          :', geo_crit, '(0 = valid for analysis)')


---
## Section 2 — Overall Crash Burden

Annual totals and month-of-year seasonality across all 43 corridors.


In [ ]:
by_year = (
    df.groupby('calendar_year')[['total_crashes', 'ksi_crashes']]
    .sum().reset_index()
)
print('Annual crash totals (all 43 corridors):')
print(by_year.to_string(index=False))
print()
print('Grand total crashes :', f"{by_year['total_crashes'].sum():,}")
print('Grand total KSI     :', f"{by_year['ksi_crashes'].sum():,}")


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 4))

ax = axes[0]
x  = by_year['calendar_year'].astype(str)
ax.bar(x, by_year['ksi_crashes'],
       label='KSI crashes', color='#c0392b', alpha=0.9)
ax.bar(x, by_year['total_crashes'] - by_year['ksi_crashes'],
       bottom=by_year['ksi_crashes'],
       label='Non-KSI crashes', color='#3498db', alpha=0.75)
ax.set_title('Annual Total Crashes (All 43 Corridors)')
ax.set_xlabel('Year')
ax.set_ylabel('Crash count')
ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda v, _: f'{int(v):,}'))
ax.legend(fontsize=9)

ax2 = axes[1]
ax2.plot(by_year['calendar_year'], by_year['ksi_crashes'],
         marker='o', color='#c0392b', linewidth=2, markersize=6)
ax2.fill_between(by_year['calendar_year'], by_year['ksi_crashes'],
                 alpha=0.15, color='#c0392b')
ax2.set_title('Annual KSI Crashes (Fatal + Serious Injury)')
ax2.set_xlabel('Year')
ax2.set_ylabel('KSI count')
for yr, val in zip(by_year['calendar_year'], by_year['ksi_crashes']):
    ax2.annotate(str(val), (yr, val), textcoords='offset points',
                 xytext=(0, 6), ha='center', fontsize=8)

plt.suptitle('Section 2 — Crash Burden by Year',
             fontsize=12, fontweight='bold', y=1.01)
plt.tight_layout()
plt.show()


In [ ]:
MONTH_NAMES = ['Jan','Feb','Mar','Apr','May','Jun',
               'Jul','Aug','Sep','Oct','Nov','Dec']
seasonality = (
    df.groupby('calendar_month')[['total_crashes', 'ksi_crashes']]
    .mean().reset_index()
)
seasonality['month_name'] = [MONTH_NAMES[m - 1] for m in seasonality['calendar_month']]
print('Average crashes per corridor-month by month of year:')
print(seasonality[['month_name', 'total_crashes', 'ksi_crashes']].to_string(index=False))


In [ ]:
fig, ax = plt.subplots(figsize=(11, 4))
ax.plot(seasonality['month_name'], seasonality['total_crashes'],
        marker='o', color='#3498db', linewidth=2, markersize=6,
        label='Total crashes (avg)')
ax2 = ax.twinx()
ax2.plot(seasonality['month_name'], seasonality['ksi_crashes'],
         marker='s', color='#c0392b', linewidth=1.5, linestyle='--',
         markersize=5, label='KSI crashes (avg)', alpha=0.85)
ax2.set_ylabel('Avg KSI per corridor-month', color='#c0392b')
ax2.tick_params(axis='y', labelcolor='#c0392b')
ax.set_title('Month-of-Year Seasonality')
ax.set_xlabel('Month')
ax.set_ylabel('Avg total crashes per corridor-month', color='#3498db')
ax.tick_params(axis='y', labelcolor='#3498db')
l1, lb1 = ax.get_legend_handles_labels()
l2, lb2 = ax2.get_legend_handles_labels()
ax.legend(l1 + l2, lb1 + lb2, fontsize=9, loc='lower right')
plt.tight_layout()
plt.show()


---
## Section 3 — Corridor Concentration

Which corridors account for the most crashes? How concentrated is the total burden?


In [ ]:
by_corridor = (
    df.groupby(['corridor_id', 'corridor_name'])[['total_crashes', 'ksi_crashes']]
    .sum().reset_index().sort_values('total_crashes', ascending=False)
)
grand_total = by_corridor['total_crashes'].sum()
top15       = by_corridor.head(15).copy()
top15_total = top15['total_crashes'].sum()
top15_share = top15_total / grand_total
top15['pct_of_total'] = (top15['total_crashes'] / grand_total * 100).round(2)

print('Grand total crashes (2018-2025):', f'{grand_total:,}')
print('Top-15 total:', f'{top15_total:,}', ' | Top-15 share:', f'{top15_share:.1%}')
print()
print(top15[['corridor_id','corridor_name','total_crashes','ksi_crashes','pct_of_total']].to_string(index=False))


In [ ]:
fig, ax = plt.subplots(figsize=(10, 6))
labels = [f"{row['corridor_name']} ({row['corridor_id']})"
          for _, row in top15.iterrows()]
values = top15['total_crashes'].values
colors = ['#c0392b' if i < 3 else '#3498db' for i in range(len(labels))]
bars = ax.barh(labels[::-1], values[::-1],
               color=colors[::-1], alpha=0.85, edgecolor='white')
for bar, val in zip(bars, values[::-1]):
    ax.text(bar.get_width() + 40,
            bar.get_y() + bar.get_height() / 2,
            f'{val:,}', va='center', fontsize=8)
ax.set_xlabel('Total recorded crashes (2018-2025)')
ax.set_title(
    f'Top 15 Corridors by Total Crashes'
    f'\n(represent {top15_share:.1%} of all {grand_total:,} corridor-recorded crashes)'
)
ax.xaxis.set_major_formatter(mticker.FuncFormatter(lambda v, _: f'{int(v):,}'))
top3_p = mpatches.Patch(color='#c0392b', alpha=0.85, label='Top 3')
rest_p = mpatches.Patch(color='#3498db', alpha=0.85, label='Rank 4-15')
ax.legend(handles=[top3_p, rest_p], fontsize=9)
plt.tight_layout()
plt.show()


---
## Section 4 — Corridor × Year Heatmap

Crash counts for every corridor × year pair. Reveals consistently high-burden corridors
and corridors with rising or falling trends.


In [ ]:
pivot_tc = (
    df.groupby(['corridor_name', 'calendar_year'])['total_crashes']
    .sum().unstack('calendar_year')
)
pivot_tc = pivot_tc.loc[pivot_tc.sum(axis=1).sort_values(ascending=False).index]
print('Heatmap pivot shape (corridors x years):', pivot_tc.shape)


In [ ]:
fig, ax = plt.subplots(figsize=(12, 10))
im = ax.imshow(pivot_tc.values, aspect='auto', cmap='YlOrRd', interpolation='nearest')
ax.set_xticks(range(len(pivot_tc.columns)))
ax.set_xticklabels(pivot_tc.columns, fontsize=9)
ax.set_yticks(range(len(pivot_tc.index)))
ax.set_yticklabels(pivot_tc.index, fontsize=8)
vmax = pivot_tc.values.max()
for r in range(pivot_tc.shape[0]):
    for c in range(pivot_tc.shape[1]):
        val = int(pivot_tc.values[r, c])
        tcol = 'white' if val > vmax * 0.55 else 'black'
        ax.text(c, r, str(val), ha='center', va='center', fontsize=7, color=tcol)
cbar = fig.colorbar(im, ax=ax, shrink=0.6)
cbar.set_label('Total recorded crashes', fontsize=9)
ax.set_title('Section 4 — Corridor × Year Total Crash Heatmap'
             '\n(sorted by 2018-2025 total burden)',
             fontsize=11, fontweight='bold')
ax.set_xlabel('Calendar Year')
ax.set_ylabel('Corridor Name')
plt.tight_layout()
plt.show()


In [ ]:
pivot_ksi = (
    df.groupby(['corridor_name', 'calendar_year'])['ksi_crashes']
    .sum().unstack('calendar_year')
)
pivot_ksi = pivot_ksi.loc[pivot_ksi.sum(axis=1).sort_values(ascending=False).index]
fig, ax = plt.subplots(figsize=(12, 10))
im = ax.imshow(pivot_ksi.values, aspect='auto', cmap='Reds', interpolation='nearest')
ax.set_xticks(range(len(pivot_ksi.columns)))
ax.set_xticklabels(pivot_ksi.columns, fontsize=9)
ax.set_yticks(range(len(pivot_ksi.index)))
ax.set_yticklabels(pivot_ksi.index, fontsize=8)
vmax_k = pivot_ksi.values.max()
for r in range(pivot_ksi.shape[0]):
    for c in range(pivot_ksi.shape[1]):
        val = int(pivot_ksi.values[r, c])
        tcol = 'white' if val > vmax_k * 0.55 else 'black'
        ax.text(c, r, str(val), ha='center', va='center', fontsize=7, color=tcol)
cbar = fig.colorbar(im, ax=ax, shrink=0.6)
cbar.set_label('KSI crashes', fontsize=9)
ax.set_title('Section 4 — Corridor × Year KSI Heatmap (sorted by KSI burden)',
             fontsize=11, fontweight='bold')
ax.set_xlabel('Calendar Year')
ax.set_ylabel('Corridor Name')
plt.tight_layout()
plt.show()


---
## Section 5 — Severity Mix

Share of each severity category across the full panel. This informs how
CMF-based treatment benefits are allocated by injury type.


In [ ]:
SEV_LABELS = {
    'fatal_crashes'              : 'Fatal (K)',
    'serious_injury_crashes'     : 'Serious Injury (A)',
    'moderate_injury_crashes'    : 'Moderate Injury (B)',
    'minor_injury_crashes'       : 'Minor Injury (C)',
    'property_damage_only_crashes': 'Property Damage Only (O)',
    'unknown_severity_crashes'   : 'Unknown',
}
sev_totals = df[list(SEV_LABELS.keys())].sum()
sev_sum    = sev_totals.sum()
sev_df = pd.DataFrame({
    'severity'  : list(SEV_LABELS.values()),
    'count'     : sev_totals.values,
    'share_pct' : (sev_totals.values / sev_sum * 100).round(2),
})
print('Severity totals (cross-check: should equal total_crashes sum)')
print('Severity col sum :', f'{sev_sum:,}')
print('total_crashes sum:', f"{df['total_crashes'].sum():,}")
print()
print(sev_df.to_string(index=False))


In [ ]:
COLORS = ['#c0392b','#e67e22','#f1c40f','#2ecc71','#3498db','#95a5a6']
fig, axes = plt.subplots(1, 2, figsize=(13, 5))

axes[0].pie(
    sev_df['count'], labels=sev_df['severity'],
    autopct='%1.1f%%', colors=COLORS, startangle=140,
    wedgeprops={'edgecolor': 'white', 'linewidth': 1.2},
    textprops={'fontsize': 8},
)
axes[0].set_title('Severity Share — All Panel (2018-2025)', fontsize=10)

yearly_sev = df.groupby('calendar_year')[list(SEV_LABELS.keys())].sum()
yearly_sev.columns = list(SEV_LABELS.values())
yearly_pct = yearly_sev.div(yearly_sev.sum(axis=1), axis=0) * 100
bottom_arr = np.zeros(len(yearly_pct))
for col, color in zip(yearly_pct.columns, COLORS):
    axes[1].bar(yearly_pct.index.astype(str), yearly_pct[col],
                bottom=bottom_arr, label=col, color=color,
                alpha=0.88, edgecolor='white')
    bottom_arr += yearly_pct[col].values
axes[1].set_title('Severity Mix by Year (% share)', fontsize=10)
axes[1].set_xlabel('Year')
axes[1].set_ylabel('Share (%)')
axes[1].legend(fontsize=7, bbox_to_anchor=(1.01, 1), loc='upper left')
plt.suptitle('Section 5 — Severity Mix', fontsize=12, fontweight='bold')
plt.tight_layout()
plt.show()


---
## Section 6 — Zero-Month Profile

How many corridor-months record **zero** total crashes or zero KSI crashes?
This directly motivates the choice of Poisson / Negative Binomial count models
over Gaussian regression.


In [ ]:
n_panel     = len(df)
zero_total  = int((df['total_crashes'] == 0).sum())
zero_ksi    = int((df['ksi_crashes']   == 0).sum())
zt_pct      = zero_total / n_panel * 100
zk_pct      = zero_ksi   / n_panel * 100

print('Zero-Month Profile')
print('  Total corridor-months         :', f'{n_panel:,}')
print('  Zero total_crashes months     :', zero_total, f'({zt_pct:.1f}%)')
print('  Non-zero total_crashes months :', n_panel - zero_total, f'({100 - zt_pct:.1f}%)')
print('  Zero ksi_crashes months       :', zero_ksi, f'({zk_pct:.1f}%)')
print('  Non-zero ksi_crashes months   :', n_panel - zero_ksi, f'({100 - zk_pct:.1f}%)')
print()
print('Overdispersion check (Var/Mean > 1.5 -> Negative Binomial preferred):')
for col in ['total_crashes', 'ksi_crashes']:
    mu  = df[col].mean()
    var = df[col].var(ddof=1)
    r   = var / mu if mu > 0 else 0.0
    tag = 'OVERDISPERSED' if r > 1.5 else 'Not strongly overdispersed'
    print(f'  {col}: mean={mu:.3f}  var={var:.3f}  Var/Mean={r:.2f}  -> {tag}')


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 4))

ax = axes[0]
ax.hist(df['total_crashes'], bins=50, color='#3498db', alpha=0.8, edgecolor='white')
ax.axvline(df['total_crashes'].mean(), color='#c0392b', linestyle='--', linewidth=1.5,
           label=f"Mean = {df['total_crashes'].mean():.1f}")
ax.set_title(f'total_crashes distribution'
             f'\nzero months: {zero_total} / {n_panel} ({zt_pct:.1f}%)')
ax.set_xlabel('Crashes per corridor-month')
ax.set_ylabel('Frequency')
ax.legend(fontsize=9)

ax2 = axes[1]
ax2.hist(df['ksi_crashes'], bins=25, color='#c0392b', alpha=0.8, edgecolor='white')
ax2.axvline(df['ksi_crashes'].mean(), color='#333', linestyle='--', linewidth=1.5,
            label=f"Mean = {df['ksi_crashes'].mean():.2f}")
ax2.set_title(f'ksi_crashes distribution'
              f'\nzero months: {zero_ksi} / {n_panel} ({zk_pct:.1f}%)')
ax2.set_xlabel('KSI crashes per corridor-month')
ax2.set_ylabel('Frequency')
ax2.legend(fontsize=9)

plt.suptitle('Section 6 — Zero-Month Profile & Count Distributions',
             fontsize=12, fontweight='bold')
plt.tight_layout()
plt.show()


---
## Section 7 — Limitations

> **This notebook is a read-only analytical exploration. It is not an engineering
> report and does not represent official City of Chicago policy.**

1. **Recorded crash burden, not exposure-adjusted risk.** The panel counts crashes
   recorded on each corridor. It does not adjust for traffic volume (AADT),
   pedestrian volumes, or other exposure measures. A high-crash corridor may
   simply carry more traffic, not be inherently more dangerous per vehicle-mile.

2. **No traffic-volume data in this version.** IDOT / CDOT AADT data has not
   been incorporated. Rate-based risk (crashes per VMT) is not computed.

3. **2026 is excluded from historical modeling.** Train/validation/test splits
   use 2018–2025 only. The 2026 forecast is a model output applied to future
   time points — it is not observed data.

4. **Corridor geometry is approximate.** Spatial assignment uses a 200-foot
   buffer tolerance. Crashes near corridor boundaries may be mis-assigned.
   Geometry validation: **PASS_WITH_WARNINGS** (0 critical failures).

5. **Monthly grain.** Sub-monthly patterns (time-of-day, day-of-week) are
   not captured at this analytical level.

6. **Decision-support only.** Final project selection authority remains with
   City staff and qualified transportation-engineering reviewers.


In [ ]:
print('=' * 60)
print('EDA SUMMARY -- all numbers computed from panel')
print('=' * 60)
print('  Rows         :', f'{n_rows:,}')
print('  Corridors    :', n_corridors)
print('  Month range  :', month_min, 'to', month_max)
print('  Total crashes:', f"{df['total_crashes'].sum():,}")
print('  Total KSI    :', f"{df['ksi_crashes'].sum():,}")
print()
print('  Top-3 corridors:')
for _, row in by_corridor.head(3).iterrows():
    pct = row['total_crashes'] / grand_total * 100
    print(f"    {row['corridor_id']} {row['corridor_name']:<22}"
          f" {row['total_crashes']:>6,}  ({pct:.1f}% of total)")
print('  Top-15 share :', f'{top15_share:.1%}')
print()
print('  Severity mix:')
for _, row in sev_df.iterrows():
    print(f"    {row['severity']:<28} {row['count']:>7,}  ({row['share_pct']:>5.1f}%)")
print()
print('  Zero total_crashes:', zero_total, '/', n_panel, f'({zt_pct:.1f}%)')
print('  Zero ksi_crashes  :', zero_ksi,   '/', n_panel, f'({zk_pct:.1f}%)')
print('=' * 60)
